# Parte 1 – Optimización Numérica (Preguntas 3 y 4)

**Redes Neuronales y Algoritmos Bioinspirados**  
Semestre 2026-01 · Universidad Nacional de Colombia

---

## Funciones de prueba elegidas

| Función | Dominio sugerido | Óptimo global |
|---------|-----------------|---------------|
| **Rosenbrock** | $[-5, 10]^d$ | $f(1,\ldots,1)=0$ |
| **Schwefel** | $[-500, 500]^d$ | $f(420.97,\ldots)\approx 0$ |

### Definiciones matemáticas

**Rosenbrock:**
$$f(\mathbf{x}) = \sum_{i=1}^{d-1}\left[100(x_{i+1}-x_i^2)^2 + (1-x_i)^2\right]$$

**Schwefel:**
$$f(\mathbf{x}) = 418.9829\,d - \sum_{i=1}^{d} x_i \sin\!\left(\sqrt{|x_i|}\right)$$

---

## Pregunta 3 – Métodos Heurísticos

Se optimizan ambas funciones en **2D** y **3D** usando:
1. Algoritmo Evolutivo (AE)
2. Optimización por Enjambre de Partículas (PSO)
3. Evolución Diferencial (DE)

## Pregunta 4 – Animaciones

Se generan GIFs que muestran el proceso de optimización (descenso por gradiente y método heurístico).


## 0. Instalación de dependencias

In [ ]:
# Instalar scipy si no está disponible (necesario para DE)
# !pip install scipy matplotlib numpy

## 1. Importaciones y funciones objetivo

In [ ]:
import numpy as np
import os
os.makedirs('../resultados', exist_ok=True)
import os
os.makedirs('../resultados', exist_ok=True)
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.optimize import differential_evolution
import imageio
import os
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)  # reproducibilidad
print('Importaciones listas.')

In [ ]:
# ─────────────────────────────────────────────
# Funciones objetivo
# ─────────────────────────────────────────────

def rosenbrock(x):
    """Función de Rosenbrock. Óptimo global: f(1,...,1) = 0."""
    x = np.asarray(x, dtype=float)
    return np.sum(100.0 * (x[1:] - x[:-1]**2)**2 + (1 - x[:-1])**2)

def schwefel(x):
    """Función de Schwefel. Óptimo global: f(420.97,...) ≈ 0."""
    x = np.asarray(x, dtype=float)
    d = len(x)
    return 418.9829 * d - np.sum(x * np.sin(np.sqrt(np.abs(x))))

# Prueba rápida
print(f'Rosenbrock en (1,1)   = {rosenbrock([1.0, 1.0]):.6f}  (esperado: 0)')
print(f'Schwefel  en (420.97, 420.97) = {schwefel([420.9687, 420.9687]):.4f}  (esperado: ~0)')

## 2. Visualización de las superficies (2D)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Rosenbrock ──
x1 = np.linspace(-2, 2, 300)
x2 = np.linspace(-1, 3, 300)
X1, X2 = np.meshgrid(x1, x2)
Z_rosen = np.array([[rosenbrock([a, b]) for a, b in zip(row1, row2)]
                    for row1, row2 in zip(X1, X2)])
cp = axes[0].contourf(X1, X2, np.log1p(Z_rosen), levels=40, cmap='viridis')
plt.colorbar(cp, ax=axes[0], label='log(1+f)')
axes[0].set_title('Rosenbrock (2D) – escala log')
axes[0].set_xlabel('$x_1$'); axes[0].set_ylabel('$x_2$')
axes[0].plot(1, 1, 'r*', markersize=12, label='Óptimo (1,1)')
axes[0].legend()

# ── Schwefel ──
x = np.linspace(-500, 500, 300)
Xg, Yg = np.meshgrid(x, x)
Z_schw = np.array([[schwefel([a, b]) for a, b in zip(row1, row2)]
                   for row1, row2 in zip(Xg, Yg)])
cp2 = axes[1].contourf(Xg, Yg, Z_schw, levels=40, cmap='plasma')
plt.colorbar(cp2, ax=axes[1], label='f(x)')
axes[1].set_title('Schwefel (2D)')
axes[1].set_xlabel('$x_1$'); axes[1].set_ylabel('$x_2$')
axes[1].plot(420.9687, 420.9687, 'r*', markersize=12, label='Óptimo (~420.97)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../resultados/superficies_2D.png', dpi=120)
plt.show()
print('Superficies guardadas.')

## 3. Algoritmo Evolutivo (AE)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Algoritmo Evolutivo – inspirado en el material de clase.
# Se guardan la población y el mejor valor por generación para las animaciones.
# ─────────────────────────────────────────────────────────────────────────────

def pob_inicial(N, d, LB, UB):
    return np.random.uniform(LB, UB, size=(N, d))

def evaluar_poblacion(Pob, f_obj):
    fitness = np.array([f_obj(ind) for ind in Pob])
    orden = np.argsort(fitness)
    return Pob[orden], fitness[orden]

def operacion_cruce(padre1, padre2):
    d = len(padre1)
    punto = np.random.randint(1, d)
    hijo1 = np.concatenate([padre1[:punto], padre2[punto:]])
    hijo2 = np.concatenate([padre2[:punto], padre1[punto:]])
    return hijo1, hijo2

def operacion_mutacion(individuo, LB, UB, prob_mut=0.3):
    nuevo = individuo.copy()
    for j in range(len(nuevo)):
        if np.random.rand() < prob_mut:
            nuevo[j] = np.random.uniform(LB[j], UB[j])
    return nuevo

def algoritmo_evolutivo(f_obj, d, LB, UB, N=50, max_gen=200,
                        fr_elite=0.2, fr_mut=0.15, semilla=None):
    """
    Algoritmo evolutivo con selección por ranking, cruce de un punto
    y mutación uniforme.

    Retorna:
        historia_pob  : lista de poblaciones por generación
        historia_best : array con el mejor valor por generación
        mejor_sol     : mejor solución encontrada
        n_eval        : número total de evaluaciones de la función objetivo
    """
    if semilla is not None:
        np.random.seed(semilla)

    LB, UB = np.array(LB), np.array(UB)
    n_elite = max(1, int(N * fr_elite))

    Pob = pob_inicial(N, d, LB, UB)
    historia_pob = []
    historia_best = []
    n_eval = 0

    for gen in range(max_gen):
        Pob, fitness = evaluar_poblacion(Pob, f_obj)
        n_eval += N
        historia_pob.append(Pob.copy())
        historia_best.append(fitness[0])

        # Élite
        nueva_pob = list(Pob[:n_elite])

        # Cruce para llenar el resto
        probs = np.arange(N, 0, -1, dtype=float)
        probs /= probs.sum()
        while len(nueva_pob) < N:
            idx1, idx2 = np.random.choice(N, 2, replace=False, p=probs)
            h1, h2 = operacion_cruce(Pob[idx1], Pob[idx2])
            nueva_pob.append(h1)
            if len(nueva_pob) < N:
                nueva_pob.append(h2)

        # Mutación
        nueva_pob = [operacion_mutacion(ind, LB, UB, fr_mut)
                     for ind in nueva_pob]
        # Preservar élite sin mutar
        nueva_pob[:n_elite] = list(Pob[:n_elite])

        Pob = np.array(nueva_pob)

    Pob, fitness = evaluar_poblacion(Pob, f_obj)
    n_eval += N
    historia_best.append(fitness[0])

    return historia_pob, np.array(historia_best), Pob[0], n_eval

print('Algoritmo Evolutivo definido.')

In [ ]:
# ── Correr AE en 2D y 3D para Rosenbrock y Schwefel ──

configs = [
    ('Rosenbrock', rosenbrock, 2, [-2, -1], [2, 3]),
    ('Rosenbrock', rosenbrock, 3, [-2, -1, -1], [2, 3, 3]),
    ('Schwefel',   schwefel,  2, [-500, -500], [500, 500]),
    ('Schwefel',   schwefel,  3, [-500, -500, -500], [500, 500, 500]),
]

resultados_ae = []
print(f"{'Función':12} {'Dim':4} {'Mejor valor':>14} {'Evaluaciones':>14}")
print('-' * 50)
for nombre, f, d, lb, ub in configs:
    hist_pob, hist_best, sol, n_eval = algoritmo_evolutivo(
        f, d, lb, ub, N=60, max_gen=300, semilla=42
    )
    resultados_ae.append((nombre, d, sol, hist_best[-1], n_eval, hist_pob, hist_best))
    print(f"{nombre:12} {d:4d} {hist_best[-1]:>14.4f} {n_eval:>14d}")

## 4. Optimización por Enjambre de Partículas (PSO)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PSO – versión cognitiva + social clásica (Inertia Weight PSO)
# ─────────────────────────────────────────────────────────────────────────────

def pso(f_obj, d, LB, UB, N=40, max_iter=300,
        w=0.7, c1=1.5, c2=1.5, semilla=None):
    """
    Optimización por Enjambre de Partículas.

    Parámetros
    ----------
    w  : peso de inercia
    c1 : coeficiente cognitivo (atracción al mejor personal)
    c2 : coeficiente social (atracción al mejor global)

    Retorna historia de posiciones, mejor valor por iteración,
    mejor solución y número de evaluaciones.
    """
    if semilla is not None:
        np.random.seed(semilla)

    LB, UB = np.array(LB, dtype=float), np.array(UB, dtype=float)
    rango = UB - LB

    # Inicialización
    pos = np.random.uniform(LB, UB, (N, d))
    vel = np.random.uniform(-rango, rango, (N, d)) * 0.1
    pbest_pos = pos.copy()               # mejor posición personal
    pbest_val = np.array([f_obj(p) for p in pos])
    gbest_idx = np.argmin(pbest_val)
    gbest_pos = pbest_pos[gbest_idx].copy()
    gbest_val = pbest_val[gbest_idx]

    historia_pos  = [pos.copy()]
    historia_best = [gbest_val]
    n_eval = N

    for it in range(max_iter):
        r1 = np.random.rand(N, d)
        r2 = np.random.rand(N, d)
        vel = (w * vel
               + c1 * r1 * (pbest_pos - pos)
               + c2 * r2 * (gbest_pos - pos))
        pos = pos + vel
        # Recortar al dominio
        pos = np.clip(pos, LB, UB)

        vals = np.array([f_obj(p) for p in pos])
        n_eval += N

        mejoras = vals < pbest_val
        pbest_pos[mejoras] = pos[mejoras]
        pbest_val[mejoras] = vals[mejoras]

        g_idx = np.argmin(pbest_val)
        if pbest_val[g_idx] < gbest_val:
            gbest_val = pbest_val[g_idx]
            gbest_pos = pbest_pos[g_idx].copy()

        historia_pos.append(pos.copy())
        historia_best.append(gbest_val)

    return historia_pos, np.array(historia_best), gbest_pos, n_eval

print('PSO definido.')

In [ ]:
# ── Correr PSO en 2D y 3D ──

resultados_pso = []
print(f"{'Función':12} {'Dim':4} {'Mejor valor':>14} {'Evaluaciones':>14}")
print('-' * 50)
for nombre, f, d, lb, ub in configs:
    hist_pos, hist_best, sol, n_eval = pso(
        f, d, lb, ub, N=50, max_iter=300, semilla=42
    )
    resultados_pso.append((nombre, d, sol, hist_best[-1], n_eval, hist_pos, hist_best))
    print(f"{nombre:12} {d:4d} {hist_best[-1]:>14.4f} {n_eval:>14d}")

## 5. Evolución Diferencial (DE) – usando `scipy`

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Evolución Diferencial (DE) con scipy.optimize.differential_evolution
# Se rastrean las evaluaciones con un contador externo.
# ─────────────────────────────────────────────────────────────────────────────

def de_con_historia(f_obj, bounds, maxiter=300, popsize=15, seed=42):
    """
    Envuelve differential_evolution de scipy y registra el histórico
    del mejor valor por generación usando el callback.
    """
    historia_best = []
    contador = {'evals': 0}

    def f_wrapped(x):
        contador['evals'] += 1
        return f_obj(x)

    def callback(xk, convergence):
        historia_best.append(f_obj(xk))

    res = differential_evolution(
        f_wrapped, bounds,
        maxiter=maxiter, popsize=popsize,
        mutation=(0.5, 1.0), recombination=0.7,
        seed=seed, callback=callback, tol=1e-12
    )
    return np.array(historia_best), res.x, res.fun, contador['evals']

print('Evolución Diferencial definida.')

In [ ]:
# ── Correr DE en 2D y 3D ──

bounds_map = {
    ('Rosenbrock', 2): [(-2, 2), (-1, 3)],
    ('Rosenbrock', 3): [(-2, 2), (-1, 3), (-1, 3)],
    ('Schwefel',   2): [(-500, 500)] * 2,
    ('Schwefel',   3): [(-500, 500)] * 3,
}

resultados_de = []
print(f"{'Función':12} {'Dim':4} {'Mejor valor':>14} {'Evaluaciones':>14}")
print('-' * 50)
for nombre, f, d, lb, ub in configs:
    bounds = bounds_map[(nombre, d)]
    hist_best, sol, val, n_eval = de_con_historia(f, bounds)
    resultados_de.append((nombre, d, sol, val, n_eval, hist_best))
    print(f"{nombre:12} {d:4d} {val:>14.4f} {n_eval:>14d}")

## 6. Convergencia comparativa

In [ ]:
# ── Gráficas de convergencia por función y dimensión ──

funciones_dims = [('Rosenbrock', 2), ('Rosenbrock', 3),
                  ('Schwefel', 2), ('Schwefel', 3)]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (fn, dim) in zip(axes.flat, funciones_dims):
    # AE
    r_ae = next(r for r in resultados_ae if r[0] == fn and r[1] == dim)
    r_pso = next(r for r in resultados_pso if r[0] == fn and r[1] == dim)
    r_de  = next(r for r in resultados_de  if r[0] == fn and r[1] == dim)

    ax.semilogy(r_ae[6],  label='AE',  color='steelblue')
    ax.semilogy(r_pso[6], label='PSO', color='darkorange')
    ax.semilogy(r_de[5],  label='DE',  color='green')
    ax.set_title(f'{fn} – {dim}D')
    ax.set_xlabel('Generación / Iteración')
    ax.set_ylabel('Mejor valor (escala log)')
    ax.legend()
    ax.grid(True, alpha=0.4)

plt.suptitle('Convergencia de métodos heurísticos', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../resultados/convergencia_heuristica.png', dpi=120)
plt.show()

## 7. Tabla resumen de resultados

In [ ]:
print('\n' + '='*70)
print(f"{'Método':6} | {'Función':10} | {'Dim':3} | {'Mejor valor':>14} | {'Evaluaciones':>12}")
print('='*70)

for r in resultados_ae:
    print(f"{'AE':6} | {r[0]:10} | {r[1]:3d} | {r[3]:>14.4f} | {r[4]:>12d}")
print('-'*70)
for r in resultados_pso:
    print(f"{'PSO':6} | {r[0]:10} | {r[1]:3d} | {r[3]:>14.4f} | {r[4]:>12d}")
print('-'*70)
for r in resultados_de:
    print(f"{'DE':6} | {r[0]:10} | {r[1]:3d} | {r[3]:>14.4f} | {r[4]:>12d}")
print('='*70)

## 8. Descenso por Gradiente (referencia para animación Q4)

In [ ]:
# -- Rosenbrock 2D --
np.random.seed(7)
x0_ros = np.random.uniform([-2, -1], [2, 3])
hist_x_gd_ros, hist_f_gd_ros, ne_gd_ros = descenso_gradiente(
    rosenbrock, x0_ros, lr=5e-4, max_iter=5000
)
print(f'GD Rosenbrock 2D -> f = {hist_f_gd_ros[-1]:.4f}, evals = {ne_gd_ros}')

# -- Schwefel 2D --
x0_sch = np.random.uniform([-500, -500], [500, 500])
hist_x_gd_sch, hist_f_gd_sch, ne_gd_sch = descenso_gradiente(
    schwefel, x0_sch, lr=1.0, max_iter=3000
)
print(f'GD Schwefel  2D -> f = {hist_f_gd_sch[-1]:.4f}, evals = {ne_gd_sch}')

# -- Rosenbrock 3D (mismo x1,x2 que 2D) --
np.random.seed(8)
x0_ros3d = np.array([x0_ros[0], x0_ros[1], np.random.uniform(-1, 3)])
hist_x_gd_ros3d, hist_f_gd_ros3d, ne_gd_ros3d = descenso_gradiente(
    rosenbrock, x0_ros3d, lr=5e-4, max_iter=5000
)
print(f'GD Rosenbrock 3D -> f = {hist_f_gd_ros3d[-1]:.4f}, evals = {ne_gd_ros3d}')

# -- Schwefel 3D (mismo x1,x2 que 2D) --
np.random.seed(9)
x0_sch3d = np.array([x0_sch[0], x0_sch[1], np.random.uniform(-500, 500)])
hist_x_gd_sch3d, hist_f_gd_sch3d, ne_gd_sch3d = descenso_gradiente(
    schwefel, x0_sch3d, lr=1.0, max_iter=3000
)
print(f'GD Schwefel  3D -> f = {hist_f_gd_sch3d[-1]:.4f}, evals = {ne_gd_sch3d}')

## 9. Pregunta 4 – Animaciones GIF

Se generan cuatro GIFs:
1. Descenso por gradiente en Rosenbrock 2D
2. Descenso por gradiente en Schwefel 2D
3. PSO en Rosenbrock 2D
4. PSO en Schwefel 2D

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Generador de GIFs con imageio.v2 (estilo frame-a-frame)
# ─────────────────────────────────────────────────────────────────────────────

def _build_contour(f_obj, xlim, ylim, N_pts=200):
    """Pre-calcula la malla de contorno para reutilizarla en todos los frames."""
    x1v = np.linspace(*xlim, N_pts)
    x2v = np.linspace(*ylim, N_pts)
    X1g, X2g = np.meshgrid(x1v, x2v)
    Zg = np.array([[f_obj([a, b]) for a, b in zip(r1, r2)]
                   for r1, r2 in zip(X1g, X2g)])
    return X1g, X2g, Zg


def create_gif_gradiente(hist_x, hist_f, f_obj, xlim, ylim, titulo,
                         levels=50, cmap='viridis', filename='../resultados/gd_anim.gif',
                         n_frames=40, duration=120):
    """
    Genera un GIF del descenso por gradiente usando imageio.v2.
    Cada frame se renderiza como PNG temporal y luego se ensamblan.
    """
    print(f'Generando GIF: {filename}...')
    X1g, X2g, Zg = _build_contour(f_obj, xlim, ylim)

    total = len(hist_x)
    indices = np.unique(np.linspace(0, total - 1, n_frames, dtype=int))
    if indices[-1] != total - 1:
        indices = np.append(indices, total - 1)

    temp_files = []
    for k, i in enumerate(indices):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        fig.suptitle(titulo, fontsize=12, fontweight='bold')

        # ── Contorno + trayectoria ──
        ax1.contourf(X1g, X2g, np.log1p(Zg), levels=levels, cmap=cmap, alpha=0.85)
        ax1.set_xlim(xlim); ax1.set_ylim(ylim)
        ax1.set_xlabel('$x_1$'); ax1.set_ylabel('$x_2$')
        ax1.set_title(f'Iter {i}')
        if i > 0:
            path = hist_x[:i+1]
            ax1.plot(path[:, 0], path[:, 1], 'r--', alpha=0.5, linewidth=1.2)
        ax1.scatter(hist_x[i, 0], hist_x[i, 1], c='red', marker='o', s=60, zorder=5)

        # ── Curva de convergencia ──
        vals_hasta_i = hist_f[:i+1]
        ax2.semilogy(vals_hasta_i, 'b-', linewidth=1.8)
        ax2.set_xlim(0, total)
        ax2.set_ylim(max(hist_f[-1] * 0.1, 1e-10), hist_f[0] * 1.1 + 1)
        ax2.set_xlabel('Iteración'); ax2.set_ylabel('f(x) – escala log')
        ax2.set_title('Convergencia')
        ax2.grid(True, alpha=0.4)

        plt.tight_layout()
        tmp = f'../resultados/_tmp_gd_{k}.png'
        plt.savefig(tmp, dpi=100)
        plt.close(fig)
        temp_files.append(tmp)

    with imageio.v2.get_writer(filename, mode='I', duration=duration) as writer:
        for f in temp_files:
            writer.append_data(imageio.v2.imread(f))
    for f in temp_files:
        os.remove(f)
    print(f'GIF guardado: {filename}')


def create_gif_pso(historia_pos, hist_best, f_obj, xlim, ylim, titulo,
                   levels=50, cmap='plasma', filename='../resultados/pso_anim.gif',
                   n_frames=40, duration=150):
    """
    Genera un GIF del PSO (nube de partículas) usando imageio.v2.
    """
    print(f'Generando GIF: {filename}...')
    X1g, X2g, Zg = _build_contour(f_obj, xlim, ylim)

    total = len(historia_pos)
    indices = np.unique(np.linspace(0, total - 1, n_frames, dtype=int))
    if indices[-1] != total - 1:
        indices = np.append(indices, total - 1)

    temp_files = []
    best_vals = np.array(hist_best)
    for k, i in enumerate(indices):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        fig.suptitle(titulo, fontsize=12, fontweight='bold')

        # ── Contorno + partículas ──
        ax1.contourf(X1g, X2g, np.log1p(Zg + 1e-8), levels=levels, cmap=cmap, alpha=0.85)
        ax1.set_xlim(xlim); ax1.set_ylim(ylim)
        ax1.set_xlabel('$x_1$'); ax1.set_ylabel('$x_2$')
        ax1.set_title(f'Iter {i}  –  mejor f = {hist_best[i]:.4f}')
        pos2d = historia_pos[i][:, :2]
        ax1.scatter(pos2d[:, 0], pos2d[:, 1], c='white', marker='x', s=25, alpha=0.8)
        vals = np.array([f_obj(p) for p in historia_pos[i]])
        best_pos = historia_pos[i][np.argmin(vals)]
        ax1.scatter(best_pos[0], best_pos[1], c='red', marker='*', s=120, zorder=5)

        # ── Curva de convergencia ──
        ax2.semilogy(best_vals[:i+1], color='darkorange', linewidth=1.8)
        ax2.set_xlim(0, total)
        ax2.set_ylim(max(best_vals[-1] * 0.1, 1e-10), best_vals[0] * 1.1 + 1)
        ax2.set_xlabel('Iteración'); ax2.set_ylabel('Mejor f(x) – escala log')
        ax2.set_title('Convergencia PSO')
        ax2.grid(True, alpha=0.4)

        plt.tight_layout()
        tmp = f'../resultados/_tmp_pso_{k}.png'
        plt.savefig(tmp, dpi=100)
        plt.close(fig)
        temp_files.append(tmp)

    with imageio.v2.get_writer(filename, mode='I', duration=duration) as writer:
        for f in temp_files:
            writer.append_data(imageio.v2.imread(f))
    for f in temp_files:
        os.remove(f)
    print(f'GIF guardado: {filename}')

print('Funciones de animación (imageio) definidas.')
def create_gif_gradiente_3d(hist_x, hist_f, f_obj, xlim, ylim, titulo,
                         filename='../resultados/gd_anim_3d.gif',
                         cmap='jet', n_frames=40, duration=150):
    """
    GIF 3D del descenso por gradiente – estilo clasico limpio.
    Mejoras vs version anterior:
      - Malla 120x120 para superficie suave
      - rstride/cstride=1 para maximo detalle
      - Vista fija (elev=28, azim=-55)
      - Fix NaN/Inf: clip de valores negativos antes de log1p
      - Esfera correctamente indexada en z_traj_all[i]
      - Mismo estilo visual que el 2D (fondo blanco, colormap jet)
    """
    print(f'Generando GIF 3D: {filename}...')
    N_MESH = 120
    x1v = np.linspace(*xlim, N_MESH)
    x2v = np.linspace(*ylim, N_MESH)
    X1g, X2g = np.meshgrid(x1v, x2v)

    # Fijar la tercera dimension en el valor inicial x0[2]
    z_fixed = hist_x[0, 2] if hist_x.shape[1] > 2 else 0
    Zg = np.array([[f_obj([a, b, z_fixed]) for a, b in zip(r1, r2)]
                   for r1, r2 in zip(X1g, X2g)])

    # Sanitizar: clip negativos para evitar NaN en log1p
    Zg_safe = np.maximum(Zg, 0)
    # Sin transformacion: valores crudos de f para preservar la forma real
    Z_plot = Zg_safe

    # Calcular limites seguros
    finite_mask = np.isfinite(Z_plot)
    if not finite_mask.any():
        print('ADVERTENCIA: todos los valores Z son no-finitos. Saltando GIF.')
        return
    z_min = np.nanmin(Z_plot[finite_mask])
    z_max = np.nanmax(Z_plot[finite_mask])
    floor_offset = z_min - (z_max - z_min) * 0.10

    total = len(hist_x)
    # Historial completo precomputado (sanitizado)
    z_traj_all = np.maximum(hist_f, 0)  # valores crudos
    indices = np.unique(np.linspace(0, total - 1, n_frames, dtype=int))
    if indices[-1] != total - 1:
        indices = np.append(indices, total - 1)

    temp_files = []

    for k, i in enumerate(indices):
        fig = plt.figure(figsize=(14, 6))
        fig.suptitle(titulo, fontsize=13, fontweight='bold')

        # ── Panel 3D ────────────────────────────────────────────────
        ax1 = fig.add_subplot(1, 2, 1, projection='3d')
        ax1.view_init(elev=28, azim=-55)

        surf = ax1.plot_surface(
            X1g, X2g, Z_plot, cmap=cmap,
            edgecolor='none', alpha=0.85,
            rstride=1, cstride=1, antialiased=True
        )

        # Contorno ligero en el piso (lineas, sin rellenar)
        ax1.contour(X1g, X2g, Z_plot, zdir='z', offset=floor_offset,
                    cmap=cmap, alpha=0.35, levels=12)

        ax1.set_xlim(xlim); ax1.set_ylim(ylim)
        ax1.set_zlim(floor_offset, z_max * 1.02)
        ax1.set_xlabel('$x_1$'); ax1.set_ylabel('$x_2$'); ax1.set_zlabel('f(x)')
        ax1.set_title(f'Iter {i} / {total-1}')

        # Trayectoria sobre la superficie hasta la iteracion i
        if i > 0:
            path = hist_x[:i+1]
            z_seg = z_traj_all[:i+1]
            ax1.plot(path[:, 0], path[:, 1], z_seg,
                     color='red', alpha=0.85, linewidth=2.0, zorder=4)
            # Sombra en el piso
            ax1.plot(path[:, 0], path[:, 1],
                     [floor_offset] * len(path),
                     'r--', alpha=0.25, linewidth=1.0, zorder=3)

        # Esfera en la posicion actual (z correctamente indexado con [i])
        ax1.scatter(hist_x[i, 0], hist_x[i, 1], z_traj_all[i],
                    color='red', s=80, zorder=5, edgecolor='black', linewidth=0.8)

        # ── Panel 2D de Convergencia ──────────────────────────────────
        ax2 = fig.add_subplot(1, 2, 2)
        vals_hasta_i = hist_f[:i+1]
        ax2.semilogy(vals_hasta_i, 'b-', linewidth=1.8)
        ax2.semilogy(i, hist_f[i], 'ro', markersize=7, zorder=5)
        ax2.set_xlim(0, total)
        ax2.set_ylim(max(hist_f[-1] * 0.1, 1e-10), hist_f[0] * 1.5)
        ax2.set_xlabel('Iteracion'); ax2.set_ylabel('f(x) -- escala log')
        ax2.set_title('Convergencia')
        ax2.grid(True, alpha=0.4)

        plt.tight_layout()
        tmp = f'../resultados/_tmp_gd3d_{k}.png'
        plt.savefig(tmp, dpi=120)
        plt.close(fig)
        temp_files.append(tmp)

    with imageio.v2.get_writer(filename, mode='I', duration=duration) as writer:
        for f in temp_files:
            writer.append_data(imageio.v2.imread(f))

    for f in temp_files:
        if os.path.exists(f): os.remove(f)
    print(f'GIF guardado: {filename}')


def create_gif_pso_3d(historia_pos, hist_best, f_obj, xlim, ylim, titulo,
                         filename='../resultados/pso_anim_3d.gif',
                         cmap='plasma', n_frames=40, duration=150):
    """
    GIF PSO 3D – estilo clasico limpio + fix NaN/Inf.
    Mejoras vs version anterior:
      - Malla 120x120 para superficie suave
      - rstride/cstride=1 para maximo detalle
      - Vista fija (elev=28, azim=-55)
      - Fix NaN/Inf: clip de valores negativos antes de log1p
      - Proyeccion de la nube al piso
    """
    print(f'Generando GIF PSO 3D: {filename}...')
    N_MESH = 120
    x1v = np.linspace(*xlim, N_MESH)
    x2v = np.linspace(*ylim, N_MESH)
    X1g, X2g = np.meshgrid(x1v, x2v)

    # Fijar la tercera dimension en el punto optimo encontrado
    val_final = np.array([f_obj(p) for p in historia_pos[-1]])
    best_final_pos = historia_pos[-1][np.argmin(val_final)]
    z_fixed = best_final_pos[2] if len(best_final_pos) > 2 else 0

    Zg = np.array([[f_obj([a, b, z_fixed]) for a, b in zip(r1, r2)]
                   for r1, r2 in zip(X1g, X2g)])

    # Sanitizar: clip negativos para evitar NaN en log1p
    Zg_safe = np.maximum(Zg, 0)
    # Sin transformacion: valores crudos de f para preservar la forma real
    Z_plot = Zg_safe

    # Calcular limites seguros
    finite_mask = np.isfinite(Z_plot)
    if not finite_mask.any():
        print('ADVERTENCIA: todos los valores Z son no-finitos. Saltando GIF.')
        return
    z_min = np.nanmin(Z_plot[finite_mask])
    z_max = np.nanmax(Z_plot[finite_mask])
    floor_offset = z_min - (z_max - z_min) * 0.10

    total = len(historia_pos)
    best_vals = np.array(hist_best)
    indices = np.unique(np.linspace(0, total - 1, n_frames, dtype=int))
    if indices[-1] != total - 1:
        indices = np.append(indices, total - 1)

    temp_files = []

    for k, i in enumerate(indices):
        fig = plt.figure(figsize=(14, 6))
        fig.suptitle(titulo, fontsize=13, fontweight='bold')

        # ── Panel 3D ────────────────────────────────────────────────
        ax1 = fig.add_subplot(1, 2, 1, projection='3d')
        ax1.view_init(elev=28, azim=-55)

        surf = ax1.plot_surface(
            X1g, X2g, Z_plot, cmap=cmap,
            edgecolor='none', alpha=0.75,
            rstride=1, cstride=1, antialiased=True
        )

        # Contorno ligero en el piso
        ax1.contour(X1g, X2g, Z_plot, zdir='z', offset=floor_offset,
                    cmap=cmap, alpha=0.35, levels=12)

        ax1.set_xlim(xlim); ax1.set_ylim(ylim)
        ax1.set_zlim(floor_offset, z_max * 1.02)
        ax1.set_xlabel('$x_1$'); ax1.set_ylabel('$x_2$'); ax1.set_zlabel('f(x)')
        ax1.set_title(f'Iter {i}  --  mejor f = {hist_best[i]:.4f}')

        # Posiciones de la nube en la iteracion i
        pos_i = historia_pos[i]
        vals_i = np.array([f_obj(p) for p in pos_i])
        z_vals = np.maximum(vals_i, 0)  # valores crudos

        idx_best = np.argmin(vals_i)
        mask = np.ones(len(pos_i), dtype=bool)
        mask[idx_best] = False

        # Particulas normales
        ax1.scatter(pos_i[mask, 0], pos_i[mask, 1], z_vals[mask],
                    c='white', marker='x', s=25, alpha=0.8, zorder=4)

        # Proyeccion de la nube en el piso
        ax1.scatter(pos_i[mask, 0], pos_i[mask, 1],
                    [floor_offset] * mask.sum(),
                    c='gray', marker='.', s=8, alpha=0.3, zorder=3)

        # Mejor particula
        ax1.scatter(pos_i[idx_best, 0], pos_i[idx_best, 1], z_vals[idx_best],
                    c='red', marker='*', s=200, zorder=6,
                    edgecolor='black', linewidth=0.8)

        # ── Panel 2D de Convergencia ──────────────────────────────────
        ax2 = fig.add_subplot(1, 2, 2)
        ax2.semilogy(best_vals[:i+1], color='darkorange', linewidth=2.0)
        ax2.semilogy(i, best_vals[i], 'ro', markersize=7, zorder=5)
        ax2.set_xlim(0, total)
        ax2.set_ylim(max(best_vals[-1] * 0.1, 1e-10), best_vals[0] * 1.5)
        ax2.set_xlabel('Iteracion'); ax2.set_ylabel('Mejor f(x) -- escala log')
        ax2.set_title('Convergencia PSO')
        ax2.grid(True, alpha=0.4)

        plt.tight_layout()
        tmp = f'../resultados/_tmp_pso3d_{k}.png'
        plt.savefig(tmp, dpi=120)
        plt.close(fig)
        temp_files.append(tmp)

    with imageio.v2.get_writer(filename, mode='I', duration=duration) as writer:
        for f in temp_files:
            writer.append_data(imageio.v2.imread(f))

    for f in temp_files:
        if os.path.exists(f): os.remove(f)
    print(f'GIF guardado: {filename}')

In [ ]:
# ── Generar GIFs de Descenso por Gradiente ──

create_gif_gradiente(
    hist_x_gd_ros, hist_f_gd_ros, rosenbrock,
    xlim=(-2.2, 2.2), ylim=(-1.2, 3.2),
    titulo='Descenso por Gradiente – Rosenbrock 2D',
    cmap='viridis', filename='../resultados/gd_rosenbrock_2D.gif'
)

create_gif_gradiente(
    hist_x_gd_sch, hist_f_gd_sch, schwefel,
    xlim=(-500, 500), ylim=(-500, 500),
    titulo='Descenso por Gradiente – Schwefel 2D',
    cmap='plasma', filename='../resultados/gd_schwefel_2D.gif'
)
create_gif_gradiente_3d(
    hist_x_gd_ros3d, hist_f_gd_ros3d, rosenbrock,
    xlim=(-2.2, 2.2), ylim=(-1.2, 3.2),
    titulo='Descenso por Gradiente – Rosenbrock 3D',
    filename='../resultados/gd_rosenbrock_3D.gif'
)

create_gif_gradiente_3d(
    hist_x_gd_sch3d, hist_f_gd_sch3d, schwefel,
    xlim=(-500, 500), ylim=(-500, 500),
    titulo='Descenso por Gradiente – Schwefel 3D',
    filename='../resultados/gd_schwefel_3D.gif'
)


In [ ]:
# ── Generar GIFs de PSO ──

r_pso_ros2 = next(r for r in resultados_pso if r[0]=='Rosenbrock' and r[1]==2)
r_pso_sch2 = next(r for r in resultados_pso if r[0]=='Schwefel'   and r[1]==2)

create_gif_pso(
    r_pso_ros2[5], r_pso_ros2[6], rosenbrock,
    xlim=(-2.2, 2.2), ylim=(-1.2, 3.2),
    titulo='PSO – Rosenbrock 2D',
    cmap='viridis', filename='../resultados/pso_rosenbrock_2D.gif'
)

create_gif_pso(
    r_pso_sch2[5], r_pso_sch2[6], schwefel,
    xlim=(-500, 500), ylim=(-500, 500),
    titulo='PSO – Schwefel 2D',
    cmap='plasma', filename='../resultados/pso_schwefel_2D.gif'
)
# ── Generar GIFs de AE ──
r_ae_ros2 = next((r for r in resultados_ae if r[0]=='Rosenbrock' and r[1]==2), None)
r_ae_sch2 = next((r for r in resultados_ae if r[0]=='Schwefel'   and r[1]==2), None)

if r_ae_ros2:
    create_gif_ae(
        r_ae_ros2[5], r_ae_ros2[6], rosenbrock,
        xlim=(-2.2, 2.2), ylim=(-1.2, 3.2),
        titulo='AE – Rosenbrock 2D',
        cmap='viridis', filename='../resultados/ae_rosenbrock_2D.gif'
    )

if r_ae_sch2:
    create_gif_ae(
        r_ae_sch2[5], r_ae_sch2[6], schwefel,
        xlim=(-500, 500), ylim=(-500, 500),
        titulo='AE – Schwefel 2D',
        cmap='plasma', filename='../resultados/ae_schwefel_2D.gif'
    )

r_pso_ros3 = next((r for r in resultados_pso if r[0]=='Rosenbrock' and r[1]==3), None)
r_pso_sch3 = next((r for r in resultados_pso if r[0]=='Schwefel'   and r[1]==3), None)

if r_pso_ros3:
    create_gif_pso_3d(
        r_pso_ros3[5], r_pso_ros3[6], rosenbrock,
        xlim=(-2.2, 2.2), ylim=(-1.2, 3.2),
        titulo='PSO – Rosenbrock 3D',
        filename='../resultados/pso_rosenbrock_3D.gif'
    )

if r_pso_sch3:
    create_gif_pso_3d(
        r_pso_sch3[5], r_pso_sch3[6], schwefel,
        xlim=(-500, 500), ylim=(-500, 500),
        titulo='PSO – Schwefel 3D',
        filename='../resultados/pso_schwefel_3D.gif'
    )


## 10. Robustez y Múltiples Corridas Experimentales

Para garantizar que los resultados no dependan puramente de la suerte con la semilla aleatoria, realizaremos 15 ejecuciones de los algoritmos heurísticos para la función de Rosenbrock en 3D y extraeremos estadísticas (promedio, desviación y mejor resultado).

In [ ]:
num_corridas = 15
d_rb = 3
lb_rb, ub_rb = [-2, -1, -1], [2, 3, 3]

res_mult_ae = []
res_mult_pso = []
res_mult_de = []

print(f'Ejecutando {num_corridas} corridas experimentales...')
for i in range(num_corridas):
    # AE
    _, hist_best_ae, _, _ = algoritmo_evolutivo(
        rosenbrock, d_rb, lb_rb, ub_rb, N=60, max_gen=300, semilla=i*10
    )
    res_mult_ae.append(hist_best_ae[-1])
    
    # PSO
    _, hist_best_pso, _, _ = pso(
        rosenbrock, d_rb, lb_rb, ub_rb, N=60, max_iter=300, semilla=i*10
    )
    res_mult_pso.append(hist_best_pso[-1])
    
    # DE
    bounds_rb = [(-2, 2), (-1, 3), (-1, 3)]
    hist_best_de, _, val_de, _ = de_con_historia(
        rosenbrock, bounds_rb, maxiter=300, popsize=15, seed=i*10
    )
    res_mult_de.append(val_de)

print('¡Corridas finalizadas!')

# --- Mostrar Estadísticas --- 
import pandas as pd
from IPython.display import display
df_stats = pd.DataFrame({
    'AE': res_mult_ae,
    'PSO': res_mult_pso,
    'DE': res_mult_de
})
print("\nEstadísticas sobre la función de Rosenbrock 3D:")
display(df_stats.agg(['mean', 'std', 'min', 'max']).T)

# Gráfico Boxplot
plt.figure(figsize=(8, 5))
df_stats.boxplot()
plt.yscale('log')
plt.title('Distribución de Mejores Valores - Rosenbrock 3D (15 Corridas)')
plt.ylabel('Valor de la Función Objetivo (Log)')
plt.savefig('../resultados/boxplot_multiples_corridas.png', dpi=120)
plt.show()


## 11. Discusión, Interpretación y Análisis Estratégico

### ¿Qué aportó el descenso por gradiente?

- **Convergencia rápida y económica** cuando la condición inicial se ubica en el valle principal del óptimo. Demuestra gran eficiencia en la explotación local.
- **Debilidades importantes señaladas en los experimentos:** 
  - Queda atrapado en **mínimos locales**, como se observó contundentemente en la función de Schwefel de alta dimensionalidad.
  - El desempeño está severamente atado a la **condición inicial aleatoria**. Si inicia muy alejado, el costo de sus evaluaciones (cálculo de gradientes) se vuelve un lastre sin éxito asegurado.
  - Tasa de aprendizaje (`lr`) requiere afinación manual crítica. Una tasa que funcionó en 2D en Rosenbrock causaba divergencia o lentitud extrema en otras iteraciones.

### ¿Qué aportaron los métodos heurísticos?

| Método | Análisis Estratégico General |
|--------|------------------------------|
| **AE** | Logró explorar un amplio espectro de la superficie de Schwefel sin caer tempranamente en el valle falso, gracias a la diversidad mantenida por el operador de mutación.|
| **PSO**| El componente de aceleración cognitiva/social le permite salir de cuencas de atracción locales gracias a la 'inercia' del peso. La gráfica de convergencia evidenció caídas abruptas cuando una partícula descubría una cuenca globalmente superior.|
| **DE** | **El ganador indiscutible** en términos de calidad de la meta. Sus permutaciones basadas en la diferencia vectorial de otros individuos lo hace sumamente efectivo frente a topologías escabrosas o valles en diagonal como los de Rosenbrock.|

> **Conclusión y Robustez:** A raíz de las experimentaciones con múltiples semillas (nuestras 15 corridas experimentales), constatamos la gran fragilidad de un resultado particular. El PSO demostró gran volatilidad de éxito (alta desviación estándar), donde pocos de sus intentos terminaban en el valle óptimo y otros eran penalizados. La Evolución Diferencial (DE) probó ser la más robusta estadísticamente. El gradiente servirá en producción únicamente como *local search* sobre las coordenadas crudas dictadas por un paso preheurístico (ej. usar un DE breve y luego inyectar la solución final al gradiente para sintonizar a precisión máquina).